# 🧠 亨創 AI 會議助理 - Qwen 2.5 7B 高管風格一鍵式 LoRA 微調
歡迎使用！本 Notebook 專為 **亨創 AI 會議助理** 量身打造，使用業界最頂級的 **Unsloth (4-bit QLoRA)** 加速架構。

### ⚡ 核心特色：
- **完全免費**：在 Google Colab 免費 T4 GPU 上即可執行。
- **極速訓練**：僅需 5~8 分鐘即可完成收斂。
- **直接匯出 GGUF**：訓練結束自動轉換為 `.gguf` 檔案，下載後直接丟進本地 `D:\project\models\` 替換即可使用！

### 步驟 1：檢查 GPU 運算環境

In [ ]:
!nvidia-smi

### 步驟 2：安裝 Unsloth 與微調加速套件

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

### 步驟 3：下載專案與訓練數據集

In [ ]:
!git clone https://github.com/TrumpWu/MeetingAssistant.git
%cd MeetingAssistant
!ls -lh TrainingData/

### 步驟 4：載入 Qwen 2.5 7B 模型並配置 LoRA 矩陣

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

max_seq_length = 4096
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ 模型與 LoRA 矩陣配置完成！")

### 步驟 5：載入黃金教材數據集 (SFT)

In [ ]:
dataset = load_dataset("json", data_files="TrainingData/training_sft.jsonl", split="train")

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        prompt = f"<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n{input_text}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(prompt)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✅ 成功載入 {len(dataset)} 組 SFT 會議教材！")

### 步驟 6：啟動微調 (約需 5~8 分鐘)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print(f"🎉 恭喜！訓練大成功，耗時：{trainer_stats.metrics.get('train_runtime', 0):.2f} 秒")

### 步驟 7：一鍵導出 GGUF 格式並直接下載到您的電腦

In [ ]:
import glob
from google.colab import files

export_dir = "hengchuang_qwen2.5_7b_gguf"
print("🚀 正在將微調模型轉換為 GGUF (Q4_K_M) 格式...")
model.save_pretrained_gguf(export_dir, tokenizer, quantization_method = "q4_k_m")

# 自動搜尋並啟動下載
gguf_files = glob.glob(f"{export_dir}/*.gguf")
if gguf_files:
    target_file = gguf_files[0]
    print(f"⬇️ 正在啟動瀏覽器下載：{target_file}")
    files.download(target_file)
    print("✨ 請將下載回來的 .gguf 檔案放進 D:\\project\\models\\ 即可享受專屬高管大腦！")
else:
    print("未找到 gguf 檔案，請檢查輸出目錄。")